# Lab 04 — Bronze ingestion

This notebook loads the prepared Online Retail Parquet batch into the pre-created Bronze Delta table.

Bronze preserves the source values—including duplicates and questionable business values—while adding technical metadata for lineage, auditing, and safe replay.

## Objectives

- load either the `initial` or `incremental` staged batch;
- retain every source column without Silver-layer cleaning;
- add stable record identifiers, batch metadata, and file metadata;
- use an insert-only Delta `MERGE` so rerunning the same batch is idempotent;
- verify row counts, lineage, uniqueness, and Delta history.

> **Structure boundary:** `lab04_00_setup` creates the Bronze table outside the production Job.
>
> **Layer boundary:** business cleaning, deduplication, and quarantine rules belong in `lab04_03_silver_quality`, not here.


## 1. Load shared configuration

The configuration notebook supplies catalog, schema, volume paths, table names, contract version, batch ID, and reset controls. Keep all Lab 4 notebooks beside one another so the relative `%run` path remains valid.

In [0]:
%run ./lab04_00_config

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

print(f"Bronze target: {bronze_table}")
print(f"Requested batch: {batch_id}")
print(f"Contract version: {contract_version}")

## 2. Select and verify the staged batch

`batch_id=initial` reads the large first delivery. `batch_id=incremental` reads the later delivery. Other staged datasets are reserved for the schema-evolution and data-quality notebooks, so this ingestion notebook rejects unsupported batch names.

In [0]:
batch_sources = {
    "initial": paths["staging_initial"],
    "incremental": paths["staging_incremental"],
}

if batch_id not in batch_sources:
    raise ValueError(
        f"Unsupported batch_id={batch_id!r}. "
        f"Choose one of {sorted(batch_sources)} for Bronze ingestion."
    )

bronze_source_path = batch_sources[batch_id]
source_files = [item for item in dbutils.fs.ls(bronze_source_path) if item.path.endswith(".parquet")]

if not source_files:
    raise FileNotFoundError(
        f"No Parquet files found under {bronze_source_path}. "
        "Run lab04_01_source_preparation.ipynb first."
    )

print(f"Source path: {bronze_source_path}")
print(f"Parquet files discovered: {len(source_files)}")

## 3. Read source values and file metadata

Databricks exposes a hidden `_metadata` struct for file-based reads. Capturing its fields in Bronze makes every record traceable to a specific file, file size, and modification time.

In [0]:
raw_batch_df = (
    spark.read
    .format("parquet")
    .load(bronze_source_path)
    .select(
        "*",
        F.col("_metadata.file_path").alias("_input_file_path"),
        F.col("_metadata.file_name").alias("_input_file_name"),
        F.col("_metadata.file_size").alias("_input_file_size"),
        F.col("_metadata.file_modification_time").alias("_input_file_modified_at"),
    )
)

required_prepared_columns = set(expected_source_columns) | {
    "_source_row_number",
    "_source_file",
    "_source_sheet",
    "_prepared_at_utc",
    "_record_hash",
}
missing_columns = sorted(required_prepared_columns - set(raw_batch_df.columns))
if missing_columns:
    raise AssertionError(f"Prepared batch is missing columns: {missing_columns}")

incoming_row_count = raw_batch_df.count()
incoming_business_duplicates = (
    incoming_row_count
    - raw_batch_df.select(*expected_source_columns).distinct().count()
)

print(f"Incoming rows: {incoming_row_count:,}")
print(f"Business duplicate rows retained in Bronze input: {incoming_business_duplicates:,}")
raw_batch_df.printSchema()

## 4. Add Bronze technical metadata

The stable `_bronze_record_id` combines workbook, sheet, and original worksheet row number. Unlike a hash of business values, it distinguishes two genuinely separate source rows even when their business fields are identical. That lets Bronze preserve source duplicates while an insert-only `MERGE` prevents duplicates caused by rerunning this notebook.

In [0]:
bronze_batch_df = (
    raw_batch_df
    .withColumn(
        "_bronze_record_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("_source_file"), F.lit("<NULL>")),
                F.coalesce(F.col("_source_sheet"), F.lit("<NULL>")),
                F.col("_source_row_number").cast("string"),
            ),
            256,
        ),
    )
    .withColumn("_batch_id", F.lit(batch_id))
    .withColumn("_source_system", F.lit("uci_online_retail"))
    .withColumn("_contract_version", F.lit(contract_version))
    .withColumn("_bronze_ingested_at", F.current_timestamp())
    .withColumn("_bronze_ingestion_date", F.current_date())
)

source_key_quality = bronze_batch_df.agg(
    F.count("*").alias("rows"),
    F.countDistinct("_bronze_record_id").alias("distinct_ids"),
    F.sum(F.col("_bronze_record_id").isNull().cast("int")).alias("null_ids"),
).first()

if source_key_quality["rows"] != source_key_quality["distinct_ids"]:
    raise AssertionError("The staged batch contains duplicate technical source-row identities.")
if source_key_quality["null_ids"] != 0:
    raise AssertionError("The staged batch contains null Bronze record IDs.")

display(bronze_batch_df.limit(20))

## 5. Verify the pre-created Bronze Delta table

The Bronze table structure is created by **`lab04_00_setup`**, which is run manually and is not part of the production Job.

This notebook performs no catalog/schema/table DDL. It only validates that the required Bronze target exists and matches the incoming technical schema.


In [0]:
if reset_demo_objects:
    raise ValueError(
        "reset_demo_objects=true is not allowed inside the production Job. "
        "Run lab04_00_setup manually when a clean structural rebuild is required."
    )

if not spark.catalog.tableExists(bronze_table):
    raise RuntimeError(
        f"Required Bronze target does not exist: {bronze_table}. "
        "Run lab04_00_setup manually before executing the Job."
    )

target_schema = {
    field.name: field.dataType.simpleString()
    for field in spark.table(bronze_table).schema.fields
}

incoming_schema = {
    field.name: field.dataType.simpleString()
    for field in bronze_batch_df.schema.fields
}

missing_in_target = sorted(set(incoming_schema) - set(target_schema))
extra_in_target = sorted(set(target_schema) - set(incoming_schema))
type_mismatches = {
    name: {
        "incoming": incoming_schema[name],
        "target": target_schema[name],
    }
    for name in sorted(set(incoming_schema) & set(target_schema))
    if incoming_schema[name] != target_schema[name]
}

if missing_in_target or extra_in_target or type_mismatches:
    raise AssertionError(
        "Bronze target schema does not match the prepared batch. "
        f"Missing in target: {missing_in_target}; "
        f"extra in target: {extra_in_target}; "
        f"type mismatches: {type_mismatches}. "
        "Run lab04_00_setup manually if the structural contract changed."
    )

print(f"✅ Pre-created Bronze target is ready: {bronze_table}")


## 6. Insert only unseen source rows with MERGE

This is an insert-only `MERGE`: a row is inserted only when its stable source-row identity is not already present. Existing Bronze rows are never updated because Bronze is an immutable record of what arrived.

In [0]:
bronze_delta = DeltaTable.forName(spark, bronze_table)
before_merge_count = spark.table(bronze_table).count()

(
    bronze_delta.alias("target")
    .merge(
        bronze_batch_df.alias("source"),
        "target._bronze_record_id = source._bronze_record_id",
    )
    .whenNotMatchedInsertAll()
    .execute()
)

after_merge_count = spark.table(bronze_table).count()
inserted_rows = after_merge_count - before_merge_count

print(f"Rows before MERGE: {before_merge_count:,}")
print(f"Rows inserted: {inserted_rows:,}")
print(f"Rows after MERGE: {after_merge_count:,}")

## 7. Prove idempotency by replaying the same batch

A production pipeline must tolerate retries. The same `MERGE` is executed a second time and the table count must remain unchanged. This is the key evidence that a job retry will not duplicate the batch.

In [0]:
count_before_replay = spark.table(bronze_table).count()

(
    DeltaTable.forName(spark, bronze_table).alias("target")
    .merge(
        bronze_batch_df.alias("source"),
        "target._bronze_record_id = source._bronze_record_id",
    )
    .whenNotMatchedInsertAll()
    .execute()
)

count_after_replay = spark.table(bronze_table).count()
if count_after_replay != count_before_replay:
    raise AssertionError(
        f"Idempotency failed: count changed from {count_before_replay} "
        f"to {count_after_replay} during replay."
    )

print("✅ Idempotency check passed: replay inserted 0 rows.")
print(f"Bronze row count remains {count_after_replay:,}.")

## 8. Validate Bronze quality and lineage

Bronze quality checks are technical rather than business-facing: IDs and file lineage must be complete, and the technical ID must be unique. Exact business duplicates are intentionally retained so Silver can handle them explicitly and transparently.

In [0]:
bronze_df = spark.table(bronze_table)
bronze_count = bronze_df.count()
business_distinct_count = bronze_df.select(*expected_source_columns).distinct().count()

technical_quality_df = bronze_df.agg(
    F.count("*").alias("bronze_rows"),
    F.countDistinct("_bronze_record_id").alias("distinct_bronze_ids"),
    F.sum(F.col("_bronze_record_id").isNull().cast("int")).alias("null_bronze_ids"),
    F.sum(F.col("_input_file_path").isNull().cast("int")).alias("null_file_paths"),
    F.countDistinct("_input_file_path").alias("processed_files"),
    F.countDistinct("_batch_id").alias("batch_count"),
)
quality = technical_quality_df.first().asDict()

if quality["bronze_rows"] != quality["distinct_bronze_ids"]:
    raise AssertionError("Bronze technical IDs are not unique.")
if quality["null_bronze_ids"] != 0 or quality["null_file_paths"] != 0:
    raise AssertionError(f"Bronze technical metadata is incomplete: {quality}")

print(f"Business duplicates retained for Silver: {bronze_count - business_distinct_count:,}")
display(technical_quality_df)
display(
    bronze_df
    .orderBy(F.col("_source_row_number"))
    .select(
        *expected_source_columns,
        "_bronze_record_id",
        "_batch_id",
        "_input_file_name",
        "_bronze_ingested_at",
    )
    .limit(20)
)

## 9. Inspect Delta transaction history

Delta history records each table operation. The two MERGE entries provide audit evidence for the original ingestion and the zero-row replay used in the idempotency test.

In [0]:
history_df = spark.sql(f"DESCRIBE HISTORY {bronze_table}")
display(
    history_df.select(
        "version",
        "timestamp",
        "operation",
        "operationParameters",
        "operationMetrics",
    ).orderBy(F.col("version").desc())
)

## 10. Completion checklist and next notebook

This notebook is complete when:

- the selected staged batch exists and its prepared schema is valid;
- the Bronze Delta table contains the expected new source rows;
- technical IDs are unique and file lineage is populated;
- the replay check reports zero inserted rows;
- Delta history shows the ingestion and replay MERGE operations.

Recommended evidence screenshots: the MERGE counts, idempotency result, technical-quality summary, and Delta history.

### Next notebook

Continue with **`lab04_03_silver_quality.ipynb`**. It will read this Bronze table, apply explicit data-quality rules, split valid and invalid rows, quarantine rejected records with reasons, and prepare the clean Silver candidate dataset.